# 04 — External validation (no retraining)

Apply the **fixed** GSE40279 elastic-net clock to an **independent** whole-blood Illumina HumanMethylation450 cohort. The estimator (`StandardScaler` + `ElasticNetCV` fitted coefficients) is loaded from disk exactly as exported by `02_model_training.ipynb`; **no parameters are refit** on the external study.

**Prerequisites (run in order, restart kernel between notebooks if needed):**

1. `01_data_loading.ipynb` — builds `data/processed/X.pkl.gz` and related files.
2. `02_model_training.ipynb` — exports `scaler.joblib`, `elasticnet_model.joblib`, and `selected_cpgs.csv` under `data/processed/`, plus internal metrics under `results/`.

**Leakage safety:** External samples are not used during GSE40279 training. Evaluation is strictly **apply frozen preprocessing + model** to external betas and compare predictions to reported ages.

## Dependencies

In [ ]:
%pip install -q pandas numpy pyarrow scikit-learn matplotlib GEOparse joblib openpyxl

## Paths

All paths are relative to the project root (`Path.cwd().parent` when this notebook runs with working directory `notebooks/`).

In [ ]:
from pathlib import Path
import shutil

PROJECT_DIR = Path.cwd().parent
DATA_DIR = PROJECT_DIR / "data"
RAW_DIR = DATA_DIR / "raw"
PROCESSED_DIR = DATA_DIR / "processed"
RESULTS_DIR = PROJECT_DIR / "results"
PUBLIC_DIR = RESULTS_DIR / "public"

EXT_SUBDIR = RAW_DIR / "GSE87571"
EXT_SUBDIR.mkdir(parents=True, exist_ok=True)
RESULTS_DIR.mkdir(parents=True, exist_ok=True)
PUBLIC_DIR.mkdir(parents=True, exist_ok=True)

EXT_GSE = "GSE87571"

## External cohort choice

We validate on **[GSE87571](https://www.ncbi.nlm.nih.gov/geo/query/acc.cgi?acc=GSE87571)** (*Continuous Aging of the Human DNA Methylome Throughout the Human Lifespan*).

**Rationale**

- **Tissue match:** genomic DNA from **whole blood** (same broad compartment as GSE40279).
- **Platform match:** **Illumina HumanMethylation450** (GPL13534), so probe identifiers align with the training feature space.
- **Outcome match:** chronological **age** is recorded per GEO sample (characteristics), enabling supervised error metrics.
- **Independence:** different center, preprocessing pipeline (SWAN beta via minfi, per series metadata), and cohort composition than GSE40279 — appropriate for an out-of-sample check.

**Note on suggested alternatives:** [GSE69724](https://www.ncbi.nlm.nih.gov/geo/query/acc.cgi?acc=GSE69724) is **mouse ChIP-seq**, not human blood methylation, and is **not** suitable here. [GSE87571](https://www.ncbi.nlm.nih.gov/geo/query/acc.cgi?acc=GSE87571) is the primary external blood 450K cohort used in this notebook.

**Data volume:** Supplementary beta matrices are large (~1.4 GB × two files). The cells below download them once into `data/raw/GSE87571/`; ensure sufficient disk space and network access.

## Load frozen model artifacts (from notebook 02)

Raises a clear error if `02_model_training.ipynb` has not been run to completion.

In [ ]:
import joblib
import pandas as pd

scaler_path = PROCESSED_DIR / "scaler.joblib"
model_path = PROCESSED_DIR / "elasticnet_model.joblib"
selected_path = PROCESSED_DIR / "selected_cpgs.csv"
internal_metrics_path = RESULTS_DIR / "model_metrics.csv"

for p in (scaler_path, model_path, selected_path):
    if not p.is_file():
        raise FileNotFoundError(
            "Missing model artifact. Run 02_model_training.ipynb first. Expected:\n" + str(p)
        )

scaler = joblib.load(scaler_path)
model = joblib.load(model_path)
selected_cpgs = pd.read_csv(selected_path)["cpg"].astype(str).tolist()

if len(selected_cpgs) != int(model.coef_.shape[0]):
    raise ValueError("selected_cpgs length does not match model.coef_")

if scaler.mean_.shape[0] != len(selected_cpgs):
    raise ValueError("Scaler feature count does not match selected_cpgs")

internal_metrics = pd.read_csv(internal_metrics_path)
print("Internal hold-out (GSE40279, from model_metrics.csv):")
display(internal_metrics)

## Download GEO metadata and supplementary methylation tables

- **SOFT / MINiML:** fetched via GEOparse into `data/raw/GSE87571/geo_soft/` (metadata + sample characteristics).
- **Betas:** series supplementary `GSE87571_matrix1of2.txt.gz` and `GSE87571_matrix2of2.txt.gz` (probes × samples), plus the optional sample spreadsheet for traceability.

In [ ]:
import urllib.request

import GEOparse

GEOparse.get_GEO(geo=EXT_GSE, destdir=str(EXT_SUBDIR / "geo_soft"), how="brief")

BASE_SUPPL = "https://ftp.ncbi.nlm.nih.gov/geo/series/GSE87nnn/GSE87571/suppl/"
downloads = {
    "GSE87571_matrix1of2.txt.gz": BASE_SUPPL + "GSE87571_matrix1of2.txt.gz",
    "GSE87571_matrix2of2.txt.gz": BASE_SUPPL + "GSE87571_matrix2of2.txt.gz",
    "GSE87571_additional_sample_chararcteristics.xlsx": BASE_SUPPL
    + "GSE87571_additional_sample_chararcteristics.xlsx",
}

for fname, url in downloads.items():
    dest = EXT_SUBDIR / fname
    if dest.is_file():
        print("Using existing", dest.name)
        continue
    print("Downloading", fname, "…")
    urllib.request.urlretrieve(url, dest)
    print("Saved", dest)

## Build sample × CpG beta matrix and chronological age labels

Probe matrices are read as **probes × samples**, then concatenated along probes and transposed to **samples × CpGs** to match `01_data_loading.ipynb` conventions.

Ages are parsed from GEO sample **characteristics** (regex on `age:` fields). This avoids silent failures if the spreadsheet layout changes.

In [ ]:
import numpy as np
import re


def read_probe_by_sample_matrix(path: Path) -> pd.DataFrame:
    df = pd.read_csv(path, sep="	", compression="gzip", low_memory=False)
    probe_col = df.columns[0]
    df = df.rename(columns={probe_col: "probe_id"}).set_index("probe_id")
    df.columns = df.columns.astype(str)
    for c in df.columns:
        df[c] = pd.to_numeric(df[c], errors="coerce")
    return df

m1 = read_probe_by_sample_matrix(EXT_SUBDIR / "GSE87571_matrix1of2.txt.gz")
m2 = read_probe_by_sample_matrix(EXT_SUBDIR / "GSE87571_matrix2of2.txt.gz")

# Normalize sample column names
m1.columns = m1.columns.astype(str).str.strip()
m2.columns = m2.columns.astype(str).str.strip()

print("m1 shape probes × samples:", m1.shape)
print("m2 shape probes × samples:", m2.shape)

# These files are split by samples, not by probes
common_probes = m1.index.intersection(m2.index)

print("m1 probes:", m1.shape[0])
print("m2 probes:", m2.shape[0])
print("common probes:", len(common_probes))

if len(common_probes) < 100_000:
    raise ValueError(
        "Too few common probes between matrix halves. "
        "Inspect GSE87571 matrix files manually."
    )

m1 = m1.loc[common_probes]
m2 = m2.loc[common_probes]

# Concatenate sample columns
beta_ps = pd.concat([m1, m2], axis=1)

# Remove duplicated sample columns if needed
if beta_ps.columns.duplicated().any():
    print("Duplicate sample columns:", beta_ps.columns.duplicated().sum())
    beta_ps = beta_ps.loc[:, ~beta_ps.columns.duplicated(keep="first")]

# Remove duplicated probes if needed
if beta_ps.index.duplicated().any():
    print("Duplicate probes:", beta_ps.index.duplicated().sum())
    beta_ps = beta_ps[~beta_ps.index.duplicated(keep="first")]

X_ext_full = beta_ps.T.astype("float32")
X_ext_full.index = X_ext_full.index.astype(str)
X_ext_full.index.name = "gsm_id"

print("External beta (samples × CpGs):", X_ext_full.shape)

gse_ext = GEOparse.get_GEO(geo=EXT_GSE, destdir=str(EXT_SUBDIR / "geo_soft"), how="brief")

gsm_ids = {str(k) for k in gse_ext.gsms.keys()}
frac_gsm_headers = float(np.mean([i in gsm_ids for i in X_ext_full.index]))
print("Fraction of matrix columns already keyed as GSM:", round(frac_gsm_headers, 3))
if frac_gsm_headers < 0.5:
    title_to_gsm = {}
    for gid, gsm in gse_ext.gsms.items():
        gid = str(gid)
        for t in gsm.metadata.get("title", []) or []:
            ts = str(t).strip()
            title_to_gsm[ts] = gid
            m = re.search(r"(?i)^X(\d+)\b", ts)
            if m:
                title_to_gsm["X" + m.group(1)] = gid
    mapped = []
    for lab in X_ext_full.index:
        if lab in gsm_ids:
            mapped.append(lab)
        elif lab in title_to_gsm:
            mapped.append(title_to_gsm[lab])
        else:
            mapped.append(lab)
    X_ext_full.index = mapped
    if X_ext_full.index.duplicated().any():
        X_ext_full = X_ext_full[~X_ext_full.index.duplicated(keep="first")]
    print("Remapped non-GSM headers; new shape:", X_ext_full.shape)


def age_from_gsm(gsm) -> float:
    chars = gsm.metadata.get("characteristics_ch1", [])
    blob = " ".join(str(x) for x in chars)
    m = re.search(r"(?i)age\s*:\s*([0-9]+(?:\.[0-9]+)?)", blob)
    if m:
        return float(m.group(1))
    compact = re.sub(r"\s+", "", blob)
    m2 = re.search(r"(?i)age:([0-9]+(?:\.[0-9]+)?)", compact)
    if m2:
        return float(m2.group(1))
    return float("nan")


age_records = {gid: age_from_gsm(gsm) for gid, gsm in gse_ext.gsms.items()}
y_ext = pd.Series(age_records, dtype=float).dropna()
y_ext.index = y_ext.index.astype(str)
y_ext.name = "age"
print("Labeled external samples (GEO age parsed):", y_ext.shape[0])

## Harmonize CpGs with the trained model (leakage-safe)

We require the **same 5 000-probe ordered list** used in GSE40279 training:

- Probes **present** in the external matrix are filled with SWAN betas and standardized with the **GSE40279 training** mean and scale (`scaler.joblib`).
- Probes **absent** from the external export (rare on 450K) are set to the training mean in standardized space (**z = 0**), equivalent to dropping only that probe's contribution when its training coefficient is non-zero.
- Samples with **any NaN** among probes that exist externally in the required list are **dropped** before prediction (conservative complete-case analysis on overlapping features).

No external labels enter fitting; we only evaluate prediction error.

In [ ]:
ext_cols = set(X_ext_full.columns.astype(str))
overlap = [c for c in selected_cpgs if c in ext_cols]
missing_probes = [c for c in selected_cpgs if c not in ext_cols]

print("Training candidate probes (selected_cpgs):", len(selected_cpgs))
print("Overlapping probes available in GSE87571:", len(overlap))
print("Training probes missing from external matrix:", len(missing_probes))
if missing_probes[:5]:
    print("Example missing:", missing_probes[:5])

common_samples = X_ext_full.index.intersection(y_ext.index).sort_values()
X_lab = X_ext_full.loc[common_samples]
y_lab = y_ext.loc[common_samples]

sub = X_lab[overlap]
before = sub.shape[0]
sub = sub.dropna(how="any")
dropped_na = before - sub.shape[0]
print("Samples after intersecting beta + age labels:", before)
print("Samples dropped (NaN in any overlapping training probe):", dropped_na)

mean_tr = scaler.mean_
scale_tr = scaler.scale_
scale_safe = np.where(scale_tr == 0, 1.0, scale_tr)

Z = np.zeros((sub.shape[0], len(selected_cpgs)), dtype=np.float64)
for j, cpg in enumerate(selected_cpgs):
    if cpg not in sub.columns:
        Z[:, j] = 0.0
        continue
    col = sub[cpg].to_numpy(dtype=np.float64)
    Z[:, j] = (col - mean_tr[j]) / scale_safe[j]

y_hat = model.predict(Z)
y_true = y_lab.loc[sub.index].to_numpy(dtype=np.float64)
print("External evaluation n:", len(y_true))

## Metrics (external cohort, frozen model)

In [ ]:
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

mae_ext = mean_absolute_error(y_true, y_hat)
rmse_ext = float(np.sqrt(mean_squared_error(y_true, y_hat)))
r2_ext = r2_score(y_true, y_hat)
pearson_ext = float(np.corrcoef(y_true, y_hat)[0, 1])

metrics_ext = pd.DataFrame(
    [
        {
            "cohort": EXT_GSE,
            "n_samples_eval": int(len(y_true)),
            "n_overlap_cpgs": len(overlap),
            "n_missing_training_cpgs_in_external": len(missing_probes),
            "samples_dropped_nan_overlap_probes": int(dropped_na),
            "MAE": mae_ext,
            "RMSE": rmse_ext,
            "R2": r2_ext,
            "Pearson_r": pearson_ext,
        }
    ]
)
display(metrics_ext)

## Figures (saved under `results/` and mirrored to `results/public/`)

In [ ]:
import matplotlib.pyplot as plt

pred_df = pd.DataFrame(
    {"observed_age": y_true, "predicted_age": y_hat}, index=sub.index
)
pred_df["residual"] = pred_df["predicted_age"] - pred_df["observed_age"]

scatter_path = RESULTS_DIR / "external_validation_scatter.png"
hist_path = RESULTS_DIR / "external_validation_residual_hist.png"

fig, ax = plt.subplots(figsize=(6, 6))
ax.scatter(pred_df["observed_age"], pred_df["predicted_age"], alpha=0.65, s=22)
lo = float(min(pred_df["observed_age"].min(), pred_df["predicted_age"].min()))
hi = float(max(pred_df["observed_age"].max(), pred_df["predicted_age"].max()))
ax.plot([lo, hi], [lo, hi], ls="--", color="gray", label="y = x")
ax.set_xlabel("Observed age (GSE87571)")
ax.set_ylabel("Predicted age (GSE40279 clock)")
ax.set_title("External validation — frozen elastic net")
ax.legend()
fig.tight_layout()
fig.savefig(scatter_path, dpi=150)
plt.show()

fig2, ax2 = plt.subplots(figsize=(8, 4))
ax2.hist(pred_df["residual"], bins=30, color="steelblue", edgecolor="white")
ax2.axvline(0.0, color="black", ls="--", lw=1)
ax2.set_xlabel("Residual (predicted − observed, years)")
ax2.set_ylabel("Count")
ax2.set_title("External residual distribution")
fig2.tight_layout()
fig2.savefig(hist_path, dpi=150)
plt.show()

## Export tables

In [ ]:
pred_out = RESULTS_DIR / "external_predictions.csv"
metrics_out = RESULTS_DIR / "external_metrics.csv"

pred_df.to_csv(pred_out, index_label="gsm_id")
metrics_ext.to_csv(metrics_out, index=False)

for name in (
    "external_validation_scatter.png",
    "external_validation_residual_hist.png",
    "external_metrics.csv",
):
    src = RESULTS_DIR / name
    if src.is_file():
        shutil.copy2(src, PUBLIC_DIR / name)

print("Wrote:", pred_out.resolve())
print("Wrote:", metrics_out.resolve())

## Interpretation

**Why external validation matters**
Internal cross-cohort performance (GSE40279 train/test split) estimates error under **shared laboratory and preprocessing context**. Transferring a clock learned in one study to another tests whether age signal **generalizes** across technical batches, normalization choices, and population shifts — the setting relevant to biomarker translation.

**Expected performance drop**
Some degradation versus internal test metrics is **normal**: the external study used **SWAN-normalized** betas from minfi (per GEO processing description) whereas GSE40279 uses series supplementary average beta values; cell-type composition, genetics, and life-course age range (here ~14–94 years) differ from the training cohort. A moderate increase in MAE/RMSE with **Pearson correlation remaining clearly positive** is often interpreted as plausible biological generalization, whereas correlation near zero suggests batch or protocol mismatch.

**Plausible drivers if generalization is limited**

- **Batch / platform effects:** dye bias, different normalization, scanner versions.
- **Preprocessing:** SWAN vs study-specific beta pipeline alters absolute beta scales; the scaler partially absorbs training-scale assumptions that may not match external data.
- **Cell composition:** whole-blood mixtures change with age and health; clocks can entangle age with leukocyte proportions.
- **Population differences:** genetic ancestry, smoking, medication, and environment shift methylation independent of chronological age.

These points motivate follow-up work (e.g., cell-type adjustment, harmonization methods) but are **not** addressed by retraining here — the scientific question in this notebook is strictly **frozen-model transport**.

## Run summary (fill by executing the notebook)

After running all cells you should record:

| Quantity | Value (from outputs above) |
|----------|----------------------------|
| External cohort | GSE87571 (whole blood, 450K) |
| Overlap CpGs | `n_overlap_cpgs` in `external_metrics.csv` |
| Internal test MAE / Pearson | columns in `model_metrics.csv` |
| External MAE / Pearson | columns in `external_metrics.csv` |

**Biological plausibility:** Compare internal vs external correlation and MAE. Substantial correlation with moderately higher error is consistent with transferable aging signal under cohort shift; correlations collapsing toward zero warrants investigating batch and preprocessing alignment before drawing biological conclusions.